# 03 · E0 (baseline) y E1 (extracción de características)
**Entrega 1 · días 3–7.** Necesita GPU: menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

Antes de ejecutar: el protocolo (`docs/02_protocolo.md`) y las hipótesis de E0 y E1 (`docs/03_hipotesis.md`) deben estar escritos.

In [ ]:
# PREPARACIÓN (igual en todos los notebooks): conecta Drive, entra a la carpeta del proyecto y deja listos los datos.
from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/yuca-comparacion'   # carpeta del proyecto dentro de su Google Drive

import os, sys
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)

from src.datos import descargar_dataset
# El .zip se guarda una sola vez en Drive; las fotos se descomprimen en el disco de Colab (más rápido).
RAIZ = str(descargar_dataset(destino='/content/datos', carpeta_zip=PROYECTO + '/data'))
os.environ['YUCA_DATOS'] = RAIZ
print('Proyecto:', os.getcwd())
print('Datos   :', RAIZ)

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU: menú Entorno de ejecución > Cambiar tipo de entorno > T4 GPU')

In [ ]:
# Configuración con la que se va a ejecutar (viene de configs/config.yaml)
import yaml, json
from src.comun import cargar_config
print(yaml.safe_dump(cargar_config(), allow_unicode=True, sort_keys=False))

## E0 · Baseline
Una red pequeña entrenada desde cero, sin conocimiento previo. Responde: *¿qué se consigue sin una arquitectura sofisticada?*

In [ ]:
from src.entrenar import ejecutar
m0 = ejecutar('E0')

In [ ]:
from IPython.display import Image as Mostrar, display
import pandas as pd
display(Mostrar('resultados/E0/curvas.png'))
display(Mostrar('resultados/E0/matriz_confusion.png'))
pd.DataFrame(m0['por_clase']).T

## E1 · Extracción de características con el Backbone A
El backbone preentrenado queda **congelado** (no cambia); solo aprende la cabeza clasificadora de 5 salidas.
Responde: *¿qué aporta el aprendizaje por transferencia?*

Para una ejecución parcial de prueba (pocas épocas) usar `ejecutar('E1', epocas=2)`: se guarda aparte como `PRUEBA_E1`.

In [ ]:
m1 = ejecutar('E1')

In [ ]:
display(Mostrar('resultados/E1/curvas.png'))
display(Mostrar('resultados/E1/matriz_confusion.png'))
pd.DataFrame(m1['por_clase']).T

## Métricas iniciales: E0 frente a E1

In [ ]:
from src import analisis
analisis.tabla_consolidada(('E0', 'E1')).T

## Registro de errores de E1
Los errores en los que el modelo estaba más seguro. Las hipótesis sobre por qué falló las escribe el equipo en la bitácora.

In [ ]:
errores = analisis.galeria_errores('E1', n=8)
display(Mostrar('resultados/E1/galeria_errores.png'))
errores[['ruta', 'real', 'predicha', 'confianza']]

In [ ]:
analisis.pares_confundidos('E1')   # confusiones más frecuentes

---
# Demostración en vivo (control del día 7)
Las celdas siguientes son las que el profesor puede pedir: cargar datos, hacer una inferencia, recalcular macro-F1 y cambiar una transformación.

### A. Cargar datos
Se arma un lote de 8 imágenes de validación y se muestra su forma: `[8 imágenes, 3 canales, 224, 224]`.

In [ ]:
from src.dataset import cargar_particiones, crear_cargador, transformaciones
cfg = cargar_config()
clases = cfg['datos']['clases']
partes = cargar_particiones()
cargador = crear_cargador(partes['validation'], RAIZ, clases, transformaciones(224), lote=8, barajar=False)
imagenes, etiquetas, _ = next(iter(cargador))
print('Forma del lote:', tuple(imagenes.shape))
print('Etiquetas:', [clases[i] for i in etiquetas.tolist()])

### B. Inferencia sobre una foto
Cambien `POSICION` para elegir otra imagen de prueba.

In [ ]:
from PIL import Image
POSICION = 0
fila = partes['test'].iloc[POSICION]
ruta = f"{RAIZ}/{fila['ruta']}"
display(Image.open(ruta).resize((300, 300)))
print('Clase real:', fila['clase'])
analisis.inferencia_una_imagen('E1', ruta)

### C. Recalcular macro-F1 a mano
Macro-F1 = promedio simple del F1 de cada clase. Cada F1 sale de la matriz de confusión:
precisión = aciertos de la clase / todo lo que el modelo llamó así; recall = aciertos / todo lo que realmente era de esa clase.

In [ ]:
import numpy as np
from src.evaluar import macro_f1_manual
matriz = np.array(m1['matriz_confusion'])   # filas = clase real, columnas = predicha
for i, c in enumerate(clases):
    vp = matriz[i, i]
    precision = vp / max(matriz[:, i].sum(), 1)
    recall = vp / max(matriz[i, :].sum(), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    print(f'{c:8} precisión={precision:.3f} recall={recall:.3f} F1={f1:.3f}')
print('Macro-F1 a mano      :', round(macro_f1_manual(matriz), 4))
print('Macro-F1 de sklearn  :', round(m1['macro_f1'], 4))

In [ ]:
# También desde el archivo de predicciones, con scikit-learn:
from sklearn.metrics import f1_score
pred = pd.read_csv('resultados/E1/predicciones_test.csv')
f1_score(pred['real'], pred['predicha'], average='macro')

### D. Cambiar una transformación
Misma foto, distinto brillo: ¿cambia la respuesta del modelo? Cambien `FACTOR` (1.0 = original).

In [ ]:
import torch
from src import modelos
from src.comun import dispositivo
from src.entrenar import definir_experimento
FACTOR = 0.5
d = definir_experimento('E1', cfg)
modelo = modelos.crear_modelo(d['modelo'], len(clases), preentrenado=False).to(dispositivo())
modelos.cargar_pesos(modelo, 'checkpoints/E1.pt', dispositivo())
modelo.eval()
for factor in (1.0, FACTOR):
    t = transformaciones(224, brillo=None if factor == 1.0 else factor)
    with torch.no_grad():
        p = torch.softmax(modelo(t(Image.open(ruta).convert('RGB')).unsqueeze(0).to(dispositivo())), 1)[0]
    print(f'brillo x{factor}: predice {clases[int(p.argmax())]} con confianza {float(p.max()):.2f}')